# Загрузка данных о бассейнах

## Импорты

In [1]:
import pandas as pd
import requests
from fuzzywuzzy import fuzz
from itertools import product
import numpy as np

/home/alx/Projects/PycharmProjects/classification-of-oil-and-gas/.venv/lib/python3.13/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


## Загрузка и сохранение

In [2]:
base_url = "https://services5.arcgis.com/33PZ8RasWNSHnkUG/ArcGIS/rest/services/Sedimentary_Basins_of_the_World/FeatureServer/0"
params = {
    'f': 'geojson',
    'where': '1=1',
    'outFields': 'BASIN_NAME,LOCATION,PET_SYS_STATUS,SUB_REGIME_GROUP,SUB_REGIME',
    'returnGeometry': 'false'
}

response = requests.get(f"{base_url}/query", params=params)
response.raise_for_status()
basin_data = response.json()

In [3]:
basins = pd.DataFrame([{"basin_name": basin.get("properties", {}).get("BASIN_NAME", ""),
                        "location": basin.get("properties", {}).get("LOCATION", ""),
                        } for basin in
                       basin_data['features']])

In [4]:
basins['location'].value_counts()

location
Onshore & Offshore    426
Onshore               290
Offshore              160
Name: count, dtype: int64

In [5]:
basins[['basin_name', 'location']].to_csv("../data/basins.csv", index=False)

## Соотнесение данных

In [6]:
train_oil_df = pd.read_csv("../data/train_oil.csv")[['Basin name']]
test_oil_df = pd.read_csv("../data/oil_test.csv")[['Basin name']]

In [7]:
basin_names_source = pd.concat([train_oil_df, test_oil_df]).fillna('unknown').drop_duplicates().reset_index(drop=True)
basin_names_source.columns = ['basin_name']
basin_names_source['basin_name'] = basin_names_source['basin_name'].str.lower()

In [8]:
basin_names_source['basin_name_key'] = basin_names_source['basin_name'].str.lower().str.replace(r'\s+', ' ', regex=True)
basins['basin_name_key'] = basins['basin_name'].str.lower().str.replace(r'\s+', ' ', regex=True)

In [9]:
basins.loc[basins['basin_name_key'] == 'anadarko - hugoton', 'basin_name_key'] = 'anadarko'
basins.loc[basins['basin_name_key'] == 'south caspian', 'basin_name_key'] = 'caspian south'
basins.loc[basins['basin_name_key'] == 'central sumatra', 'basin_name_key'] = 'sumatra central'
basins.loc[basins['basin_name_key'] == 'north sumatra - mergui', 'basin_name_key'] = 'sumatra north'
basins.loc[basins['basin_name_key'] == 'south sumatra', 'basin_name_key'] = 'sumatra south'
basins.loc[basins['basin_name_key'] == 'southern north sea - anglo-dutch', 'basin_name_key'] = 'north sea southern'
basins.loc[basins['basin_name_key'] == 'northern north sea', 'basin_name_key'] = 'north sea northern'
basins.loc[basins['basin_name_key'] == 'mid north sea high', 'basin_name_key'] = 'north sea central'

In [10]:
pairs = list(product(basin_names_source.index, basins.index))

In [11]:
matches = [(i, j, fuzz.ratio(basin_names_source.loc[i, 'basin_name_key'], basins.loc[j, 'basin_name_key'])) 
           for i, j in pairs 
           if fuzz.ratio(basin_names_source.loc[i, 'basin_name_key'], basins.loc[j, 'basin_name_key']) >= 75]

In [12]:
matched = pd.DataFrame(matches, columns=['idx1', 'idx2', 'score'])

In [13]:
result = (basin_names_source.loc[matched['idx1'], ['basin_name']]
          .reset_index(drop=True)
          .join(basins.loc[matched['idx2']].reset_index(drop=True), 
                rsuffix='_basins'))

In [14]:
result.drop(result[((result['basin_name'] == 'nile delta') & (result['basin_name_key'] == 'niger delta'))].index, inplace=True)
result.drop(result[((result['basin_name'] == 'niger delta') & (result['basin_name_key'] == 'nile delta'))].index, inplace=True)
result.drop(result[((result['basin_name'] == 'tarakan') & (result['basin_name_key'] == 'arakan'))].index, inplace=True)
result.drop(result[((result['basin_name'] == 'paris') & (result['basin_name_key'] == 'parecis'))].index, inplace=True)
result.drop(result[((result['basin_name'] == 'carnarvon') & (result['basin_name_key'] == 'south carnarvon'))].index, inplace=True)
result.drop(result[((result['basin_name'] == 'java northwest') & (result['basin_name_key'] == 'northwest'))].index, inplace=True)
result.drop(result[((result['basin_name'] == 'paris') & (result['basin_name_key'] == 'parecis'))].index, inplace=True)
result.drop(result[((result['basin_name'] == 'powder river') & (result['basin_name_key'] == 'copper river'))].index, inplace=True)
result.drop(result[((result['basin_name'] == 'williston') & (result['basin_name_key'] == 'biliton'))].index, inplace=True)
result.drop(result[((result['basin_name'] == 'permian') & (result['basin_name_key'] == 'erlian'))].index, inplace=True)
result.drop(result[((result['basin_name'] == 'bonaparte') & (result['basin_name_key'] == 'bonaire'))].index, inplace=True)

In [15]:
basins_mapped = result[['basin_name', 'location']]

In [16]:
col = 'location'
conditions = [basins_mapped[col] == 'Onshore',
              basins_mapped[col] == 'Offshore',
              basins_mapped[col] == 'Onshore & Offshore']
choices= ["onshore", 'offshore', 'onshore-offshore']

In [17]:
basins_mapped[col] = np.select(conditions, choices, default='')

In [18]:
basins_mapped[['basin_name', 'location']].to_csv("../data/basins_mapped.csv", index=False)

In [19]:
basins_mapped.shape

(67, 2)